In [1]:
import pandas as pd
import numpy as np
import random

# ==========================================
# STEP 1: GENERATE SYNTHETIC LOAN PORTFOLIO
# ==========================================
np.random.seed(42)
n_loans = 10000

# 1. Base Loan Characteristics
loan_data = pd.DataFrame({
    'Loan_ID': range(1, n_loans + 1),
    'Loan_Amount': np.random.uniform(10000, 500000, n_loans).round(2),
    'Interest_Rate': np.random.uniform(0.03, 0.15, n_loans).round(4),
    'Credit_Score': np.random.randint(300, 850, n_loans), # FICO style
    'LTV_Ratio': np.random.uniform(0.2, 1.2, n_loans).round(4), # Loan-to-Value
    'Remaining_Term_Months': np.random.randint(1, 360, n_loans),
    'Days_Past_Due': np.random.choice([0, 0, 0, 0, 0, 15, 30, 60, 90, 120], n_loans) # Skewed towards 0
})

# 2. Credit Line Characteristics (for EAD calculation)
# Drawn Amount cannot exceed Loan Amount
loan_data['Drawn_Amount'] = loan_data['Loan_Amount'] * np.random.uniform(0.5, 1.0, n_loans)
loan_data['Undrawn_Limit'] = loan_data['Loan_Amount'] - loan_data['Drawn_Amount']

# 3. Origination PD (Probability of Default at the time the loan was issued)
# Lower credit score = higher PD. We'll map this roughly for IFRS 9 SICR testing later.
def map_origination_pd(score):
    if score >= 750: return 0.005
    elif score >= 650: return 0.02
    elif score >= 550: return 0.05
    else: return 0.12

loan_data['Origination_PD'] = loan_data['Credit_Score'].apply(map_origination_pd)

print("✅ Synthetic Loan Portfolio successfully generated!")
print(f"Total Loans: {len(loan_data)}")
print("\n📊 First 5 rows of the Portfolio:")
display(loan_data.head())

✅ Synthetic Loan Portfolio successfully generated!
Total Loans: 10000

📊 First 5 rows of the Portfolio:


,Loan_ID,Loan_Amount,Interest_Rate,Credit_Score,LTV_Ratio,Remaining_Term_Months,Days_Past_Due,Drawn_Amount,Undrawn_Limit,Origination_PD
0,1,193524.66,0.0748,557,0.7874,27,0,105796.869314,87727.790686,0.05
1,2,475850.01,0.0699,687,1.1570,282,30,299930.779467,175919.230533,0.02
2,3,368677.03,0.0511,344,0.7739,279,0,205677.238097,162999.791903,0.12
3,4,303342.66,0.1029,568,0.3587,120,0,190011.463054,113331.196946,0.05
4,5,86449.13,0.0872,610,0.7175,253,60,44285.468014,42163.661986,0.05


In [ ]:
#
#Calculate EAD (Exposure at Default)
#Under IFRS 9, EAD is not just the current outstanding balance. For revolving 
#credit facilities (like credit cards or overdrafts), we must include the portion 
#the borrower might draw down before defaulting. This is done using a Credit Conversion Factor (CCF).
#

In [2]:
# ==========================================
# STEP 2: CALCULATE EAD (EXPOSURE AT DEFAULT)
# ==========================================

# Define the Credit Conversion Factor (CCF)
# For this PoC, we will use a standard Basel/IFRS simplified CCF of 75% for undrawn commitments.
CCF = 0.75

# EAD = Drawn Amount + (Undrawn Limit * CCF)
loan_data['EAD'] = loan_data['Drawn_Amount'] + (loan_data['Undrawn_Limit'] * CCF)

print("✅ EAD successfully calculated!")
print(f"Average EAD per loan: ${loan_data['EAD'].mean():,.2f}")
print(f"Total Portfolio EAD: ${loan_data['EAD'].sum():,.2f}")

✅ EAD successfully calculated!
Average EAD per loan: $236,377.90
Total Portfolio EAD: $2,363,779,034.30


In [ ]:
#
#Before we build the machine learning model in Day 2, we need a baseline.
#PD: We will use a simplified logistic mapping based on the Credit Score 
#    to establish a baseline Point-in-Time (PiT) PD.
#LGD: Loss Given Default is heavily driven by collateral. We will map the
#    LTV (Loan-to-Value) ratio to an LGD. (Higher LTV = less collateral coverage = higher LGD).
#


In [3]:
# ==========================================
# STEP 3: DEFINE BASELINE PD AND LGD
# ==========================================

# 1. Baseline PD Mapping (Simplified Logistic Proxy based on Credit Score)
# In Day 2, we will replace this with a proper scikit-learn Logistic Regression model.
def calculate_baseline_pd(score):
    # Simplified inverse relationship: higher score = lower PD
    # Using a rough exponential decay for demonstration
    return 0.5 * np.exp(-0.01 * (score - 300))

loan_data['Baseline_PD'] = loan_data['Credit_Score'].apply(calculate_baseline_pd)
# Cap PD between 0.1% and 99%
loan_data['Baseline_PD'] = loan_data['Baseline_PD'].clip(0.001, 0.99)

# 2. LGD Mapping based on LTV (Loan-to-Value)
# If LTV > 1.0 (underwater loan), LGD is very high (e.g., 80-100%).
# If LTV < 0.6 (strong collateral), LGD is low (e.g., 20-30%).
def calculate_lgd(ltv):
    if ltv >= 1.0:
        return np.random.uniform(0.80, 1.00)
    elif ltv >= 0.8:
        return np.random.uniform(0.50, 0.80)
    elif ltv >= 0.6:
        return np.random.uniform(0.30, 0.50)
    else:
        return np.random.uniform(0.10, 0.30)

loan_data['LGD'] = loan_data['LTV_Ratio'].apply(calculate_lgd)

print("✅ Baseline PD and LGD successfully calculated!")
print(f"Average Baseline PD: {loan_data['Baseline_PD'].mean():.4f} ({loan_data['Baseline_PD'].mean()*100:.2f}%)")
print(f"Average LGD: {loan_data['LGD'].mean():.4f} ({loan_data['LGD'].mean()*100:.2f}%)")

✅ Baseline PD and LGD successfully calculated!
Average Baseline PD: 0.0907 (9.07%)
Average LGD: 0.4689 (46.89%)


In [ ]:
#
#Now we write the fundamental IFRS 9 formula: ECL = EAD × PD × LGD.
#Note: For a 12-month ECL (Stage 1), we use the 12-month PD. 
#For Lifetime ECL (Stage 2 & 3), we use the Lifetime PD. For Day 1, we will 
#calculate the basic formula, and in Day 3 we will apply the specific 12-month vs. Lifetime adjustments.
#

In [4]:
# ==========================================
# STEP 4: CORE ECL CALCULATION ENGINE
# ==========================================

def calculate_ecl(ead, pd, lgd):
    """
    Core IFRS 9 ECL Formula.
    ECL = Exposure at Default * Probability of Default * Loss Given Default
    """
    return ead * pd * lgd

# Apply the function to calculate the ECL for each loan
# For Day 1, we are calculating a generic "Base ECL" using the Baseline PD.
loan_data['Base_ECL'] = loan_data.apply(
    lambda row: calculate_ecl(row['EAD'], row['Baseline_PD'], row['LGD']), 
    axis=1
)

print("✅ Core ECL Engine successfully executed!")
print(f"Total Portfolio Base ECL: ${loan_data['Base_ECL'].sum():,.2f}")
print(f"Average ECL per loan: ${loan_data['Base_ECL'].mean():,.2f}")

# ==========================================
# STEP 5: PRELIMINARY PORTFOLIO AGGREGATION
# ==========================================
portfolio_summary = pd.DataFrame({
    'Metric': ['Total Loans', 'Total EAD', 'Total Base ECL', 'Avg Credit Score', 'Avg LTV'],
    'Value': [
        len(loan_data),
        loan_data['EAD'].sum(),
        loan_data['Base_ECL'].sum(),
        loan_data['Credit_Score'].mean(),
        loan_data['LTV_Ratio'].mean()
    ]
})

print("\n📊 Portfolio Summary:")
display(portfolio_summary)

# Save to CSV for Day 2 & 3
loan_data.to_csv('day1_ifrs9_loan_portfolio.csv', index=False)
print("\n✅ Day 1 Portfolio exported to 'day1_ifrs9_loan_portfolio.csv'")

✅ Core ECL Engine successfully executed!
Total Portfolio Base ECL: $102,068,292.86
Average ECL per loan: $10,206.83

📊 Portfolio Summary:


,Metric,Value
0,Total Loans,1.000000e+04
1,Total EAD,2.363779e+09
2,Total Base ECL,1.020683e+08
3,Avg Credit Score,5.764276e+02
4,Avg LTV,6.983673e-01



✅ Day 1 Portfolio exported to 'day1_ifrs9_loan_portfolio.csv'


In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Load Day 1 data
loan_data = pd.read_csv('day1_ifrs9_loan_portfolio.csv')

# ==========================================
# STEP 1: GENERATE DEFAULT LABELS (Synthetic History)
# ==========================================
# We use the Baseline_PD calculated in Day 1 as the probability of default for each loan
# np.random.binomial generates 1 (default) or 0 (no default) based on that probability
np.random.seed(42)
loan_data['Default_Flag'] = np.random.binomial(1, loan_data['Baseline_PD'])

print(f"✅ Synthetic Default Flags generated.")
print(f"Portfolio Default Rate: {loan_data['Default_Flag'].mean()*100:.2f}%\n")

# ==========================================
# STEP 2: TRAIN LOGISTIC REGRESSION PD MODEL
# ==========================================
# Features: Credit Score and LTV Ratio are the strongest predictors of default
X = loan_data[['Credit_Score', 'LTV_Ratio']]
y = loan_data['Default_Flag']

# Split data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the Logistic Regression model
# Note: In production, we would scale features, but for this PoC, raw values work fine for demonstration
pd_model = LogisticRegression(random_state=42, max_iter=1000)
pd_model.fit(X_train, y_train)

# Evaluate model using AUC-ROC (Area Under the Receiver Operating Characteristic Curve)
# AUC > 0.7 is generally acceptable for credit risk models
y_pred_proba = pd_model.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_pred_proba)

print(f"✅ Logistic Regression PD Model trained successfully!")
print(f"Model Performance (AUC-ROC): {auc_score:.4f}")
print("*(Note: AUC ranges from 0.5 to 1.0. Higher is better. >0.7 indicates good predictive power.)*\n")

# ==========================================
# STEP 3: APPLY MODEL TO PREDICT POINT-IN-TIME (PiT) PD
# ==========================================
# Predict the Probability of Default for the entire portfolio using the new ML model
loan_data['ML_Pit_PD'] = pd_model.predict_proba(X)[:, 1]

# Cap PD between 0.1% and 99% to prevent extreme outliers
loan_data['ML_Pit_PD'] = loan_data['ML_Pit_PD'].clip(0.001, 0.99)

print("✅ Point-in-Time (PiT) PD successfully predicted using ML model.")
print(f"Average ML PiT PD: {loan_data['ML_Pit_PD'].mean():.4f} ({loan_data['ML_Pit_PD'].mean()*100:.2f}%)")

✅ Synthetic Default Flags generated.
Portfolio Default Rate: 8.75%

✅ Logistic Regression PD Model trained successfully!
Model Performance (AUC-ROC): 0.8599
*(Note: AUC ranges from 0.5 to 1.0. Higher is better. >0.7 indicates good predictive power.)*

✅ Point-in-Time (PiT) PD successfully predicted using ML model.
Average ML PiT PD: 0.0884 (8.84%)


In [ ]:
#
#IFRS 9 requires us to adjust the baseline PD based on future economic scenarios (Base, Upside, Downside). 
#We will simulate this by applying a multiplicative overlay based on synthetic macroeconomic variables
#(e.g., Unemployment Rate change).
#

In [6]:
# ==========================================
# STEP 2: MACROECONOMIC OVERLAYS (FORWARD-LOOKING INFORMATION)
# ==========================================

# Define 3 IFRS 9 Scenarios with their probabilities and macroeconomic shifts
scenarios = {
    'Base': {'weight': 0.60, 'unemployment_shift': 0.0, 'pd_multiplier': 1.0},
    'Upside': {'weight': 0.20, 'unemployment_shift': -0.5, 'pd_multiplier': 0.85}, # Lower unemployment = lower PD
    'Downside': {'weight': 0.20, 'unemployment_shift': 1.5, 'pd_multiplier': 1.35}  # Higher unemployment = higher PD
}

# Apply overlays to the ML PiT PD
for scenario_name, params in scenarios.items():
    # Adjusted PD = Baseline ML PD * Macroeconomic Multiplier
    # Cap at 99% to ensure it remains a valid probability
    loan_data[f'PD_{scenario_name}'] = (loan_data['ML_Pit_PD'] * params['pd_multiplier']).clip(upper=0.99)

# Calculate the Probability-Weighted Expected PD (The core IFRS 9 requirement)
loan_data['Weighted_ECL_PD'] = (
    loan_data['PD_Base'] * scenarios['Base']['weight'] +
    loan_data['PD_Upside'] * scenarios['Upside']['weight'] +
    loan_data['PD_Downside'] * scenarios['Downside']['weight']
)

print("✅ Macroeconomic Overlays applied successfully!")
print("\n📊 Scenario PD Comparison (First 5 loans):")
display(loan_data[['Loan_ID', 'Credit_Score', 'ML_Pit_PD', 'PD_Base', 'PD_Upside', 'PD_Downside', 'Weighted_ECL_PD']].head())

✅ Macroeconomic Overlays applied successfully!

📊 Scenario PD Comparison (First 5 loans):


,Loan_ID,Credit_Score,ML_Pit_PD,PD_Base,PD_Upside,PD_Downside,Weighted_ECL_PD
0,1,557,0.038021,0.038021,0.032318,0.051329,0.039542
1,2,687,0.009605,0.009605,0.008164,0.012966,0.009989
2,3,344,0.316746,0.316746,0.269234,0.427607,0.329415
3,4,568,0.030059,0.030059,0.025550,0.040579,0.031261
4,5,610,0.020567,0.020567,0.017482,0.027765,0.021390


In [7]:
# ==========================================
# STEP 3: RECALCULATE ECL WITH FORWARD-LOOKING PD
# ==========================================

# Re-apply the core ECL formula using the new Weighted PD
loan_data['Forward_Looking_ECL'] = loan_data.apply(
    lambda row: calculate_ecl(row['EAD'], row['Weighted_ECL_PD'], row['LGD']), 
    axis=1
)

# Compare Day 1 Base ECL vs Day 2 Forward-Looking ECL
total_base_ecl = loan_data['Base_ECL'].sum()
total_fl_ecl = loan_data['Forward_Looking_ECL'].sum()
ecl_increase = total_fl_ecl - total_base_ecl

print("✅ Forward-Looking ECL successfully calculated!")
print(f"\n📊 Portfolio ECL Impact Analysis:")
print(f"Day 1 Base ECL (Static):       ${total_base_ecl:,.2f}")
print(f"Day 2 Forward-Looking ECL:     ${total_fl_ecl:,.2f}")
print(f"IFRS 9 FLI Adjustment (Delta): ${ecl_increase:,.2f} ({(ecl_increase/total_base_ecl)*100:+.2f}%)")
print("\n*(Note: The Downside scenario typically forces the overall portfolio ECL to increase, reflecting prudent risk management.)*")

# Export updated data for Day 3
loan_data.to_csv('day2_ifrs9_ml_and_macro_ecl.csv', index=False)
print("\n✅ Day 2 Portfolio exported to 'day2_ifrs9_ml_and_macro_ecl.csv'")

✅ Forward-Looking ECL successfully calculated!

📊 Portfolio ECL Impact Analysis:
Day 1 Base ECL (Static):       $102,068,292.86
Day 2 Forward-Looking ECL:     $106,765,249.58
IFRS 9 FLI Adjustment (Delta): $4,696,956.72 (+4.60%)

*(Note: The Downside scenario typically forces the overall portfolio ECL to increase, reflecting prudent risk management.)*

✅ Day 2 Portfolio exported to 'day2_ifrs9_ml_and_macro_ecl.csv'


In [8]:
#
# A +4.60% increase in the ECL provision due to the Forward-Looking Information (FLI) overlay is a 
# highly realistic outcome. It perfectly demonstrates the "prudent" nature of IFRS 9, 
# where banks must hold more capital today to account for potential economic downturns tomorrow.
#

#================
# Day3
#================


#IFRS 9 Staging Logic (SICR & Default)
#Under IFRS 9, loans are classified into three stages based on credit quality. 
#This determines whether we book 12-month ECL or Lifetime ECL.
#Stage 1 (Performing): 12-month ECL.
#Stage 2 (Underperforming/SICR): Lifetime ECL. Triggered by a Significant Increase in Credit Risk (SICR).
#Stage 3 (Non-performing/Default): Lifetime ECL. Triggered by default (e.g., 90+ days past due).

In [9]:
import pandas as pd
import numpy as np

# Load Day 2 data
loan_data = pd.read_csv('day2_ifrs9_ml_and_macro_ecl.csv')

# ==========================================
# STEP 1: IFRS 9 STAGING LOGIC
# ==========================================

# 1. Define Default (Stage 3): 90+ Days Past Due OR actual default flag
loan_data['Is_Default'] = (loan_data['Days_Past_Due'] >= 90) | (loan_data['Default_Flag'] == 1)

# 2. Define SICR (Significant Increase in Credit Risk) for Stage 2
# Rule: If current Weighted PD is more than double the Origination PD, credit risk has significantly increased.
loan_data['Is_SICR'] = loan_data['Weighted_ECL_PD'] > (loan_data['Origination_PD'] * 2)

# 3. Assign Stages
def assign_stage(row):
    if row['Is_Default']:
        return 3
    elif row['Is_SICR'] or row['Days_Past_Due'] >= 30:
        return 2
    else:
        return 1

loan_data['IFRS9_Stage'] = loan_data.apply(assign_stage, axis=1)

print("✅ IFRS 9 Staging successfully applied!")
print("\n📊 Portfolio Distribution by Stage:")
print(loan_data['IFRS9_Stage'].value_counts().sort_index())

✅ IFRS 9 Staging successfully applied!

📊 Portfolio Distribution by Stage:
IFRS9_Stage
1    5002
2    2322
3    2676
Name: count, dtype: int64


In [10]:
#
#Calculate 12-Month vs. Lifetime ECL
#Now we calculate the final regulatory ECL.
#Stage 1 uses 12-month ECL (the expected loss from defaults possible in the next 12 months).
#Stage 2 & 3 use Lifetime ECL (the expected loss over the entire remaining life of the loan).
#

In [11]:
# ==========================================
# STEP 2: CALCULATE 12-MONTH vs LIFETIME ECL
# ==========================================

# Proxy for 12-Month PD: 
# For simplicity in this PoC, we approximate 12-month PD by scaling the Lifetime PD 
# based on the remaining term. (12 months / Remaining Term).
loan_data['Twelve_Month_PD'] = loan_data['Weighted_ECL_PD'] * (12 / loan_data['Remaining_Term_Months'].clip(lower=12))

# Calculate Stage-Specific ECL
def calculate_stage_ecl(row):
    ead = row['EAD']
    lgd = row['LGD']
    
    if row['IFRS9_Stage'] == 1:
        # Stage 1: 12-month ECL
        return ead * row['Twelve_Month_PD'] * lgd
    else:
        # Stage 2 & 3: Lifetime ECL (using the full Weighted Probability-Weighted PD)
        return ead * row['Weighted_ECL_PD'] * lgd

loan_data['Final_IFRS9_ECL'] = loan_data.apply(calculate_stage_ecl, axis=1)

# ==========================================
# STEP 3: PORTFOLIO AGGREGATION & REPORTING
# ==========================================

# Aggregate by Stage
stage_summary = loan_data.groupby('IFRS9_Stage').agg(
    Loan_Count=('Loan_ID', 'count'),
    Total_EAD=('EAD', 'sum'),
    Total_ECL=('Final_IFRS9_ECL', 'sum')
).reset_index()

stage_summary['ECL_as_%_of_EAD'] = (stage_summary['Total_ECL'] / stage_summary['Total_EAD']) * 100

print("✅ Final IFRS 9 ECL successfully calculated!")
print("\n📊 IFRS 9 Regulatory Reporting Summary:")
display(stage_summary)

# Save final output
loan_data.to_csv('day3_ifrs9_final_staging_ecl.csv', index=False)

✅ Final IFRS 9 ECL successfully calculated!

📊 IFRS 9 Regulatory Reporting Summary:


,IFRS9_Stage,Loan_Count,Total_EAD,Total_ECL,ECL_as_%_of_EAD
0,1,5002,1.132851e+09,3.366284e+06,0.297151
1,2,2322,4.767919e+08,2.844970e+07,5.966902
2,3,2676,7.541361e+08,5.580656e+07,7.400065


In [12]:
# ==========================================
# STEP 4: MODEL GOVERNANCE & VALIDATION SUMMARY
# ==========================================

total_ecl = loan_data['Final_IFRS9_ECL'].sum()
stage1_ecl = loan_data[loan_data['IFRS9_Stage'] == 1]['Final_IFRS9_ECL'].sum()
stage2_ecl = loan_data[loan_data['IFRS9_Stage'] == 2]['Final_IFRS9_ECL'].sum()
stage3_ecl = loan_data[loan_data['IFRS9_Stage'] == 3]['Final_IFRS9_ECL'].sum()

governance_report = f"""
======================================================================
       MODEL RISK MANAGEMENT & IFRS 9 GOVERNANCE REPORT
======================================================================
1. MODEL OVERVIEW
   - Model Type: IFRS 9 Expected Credit Loss (ECL) Engine
   - PD Model: Logistic Regression (Scikit-Learn)
   - Key Features: Credit Score, LTV Ratio
   - Validation Metric: AUC-ROC (Assessed during Day 2)

2. IFRS 9 STAGING & ECL PROVISIONING
   - Total Portfolio ECL Provision: ${total_ecl:,.2f}
   - Stage 1 (Performing - 12m ECL): ${stage1_ecl:,.2f}
   - Stage 2 (SICR - Lifetime ECL):  ${stage2_ecl:,.2f}
   - Stage 3 (Default - Lifetime ECL):${stage3_ecl:,.2f}

3. FORWARD-LOOKING INFORMATION (FLI) OVERLAY
   - Scenarios Applied: Base (60%), Upside (20%), Downside (20%)
   - Macroeconomic Variable: Unemployment Rate Shift
   - Impact: Downside scenario increased portfolio ECL by ~4.6%

4. MODEL LIMITATIONS & ASSUMptions
   - Synthetic Data: Model trained on synthetically generated retail loan data.
   - Static LGD: LGD was mapped via LTV buckets; in production, a Beta regression model is recommended.
   - CCF Assumption: A flat 75% Credit Conversion Factor was applied to undrawn limits.

5. RECOMMENDATIONS FOR PRODUCTION
   - Implement automated quarterly recalibration of the Logistic Regression PD model.
   - Integrate live macroeconomic feeds (e.g., IMF, Central Bank) for dynamic FLI overlays.
   - Conduct annual independent model validation by the Model Risk Management (MRM) team.
======================================================================
"""

print(governance_report)


       MODEL RISK MANAGEMENT & IFRS 9 GOVERNANCE REPORT
1. MODEL OVERVIEW
   - Model Type: IFRS 9 Expected Credit Loss (ECL) Engine
   - PD Model: Logistic Regression (Scikit-Learn)
   - Key Features: Credit Score, LTV Ratio
   - Validation Metric: AUC-ROC (Assessed during Day 2)

2. IFRS 9 STAGING & ECL PROVISIONING
   - Total Portfolio ECL Provision: $87,622,546.37
   - Stage 1 (Performing - 12m ECL): $3,366,283.81
   - Stage 2 (SICR - Lifetime ECL):  $28,449,701.54
   - Stage 3 (Default - Lifetime ECL):$55,806,561.01

3. FORWARD-LOOKING INFORMATION (FLI) OVERLAY
   - Scenarios Applied: Base (60%), Upside (20%), Downside (20%)
   - Macroeconomic Variable: Unemployment Rate Shift
   - Impact: Downside scenario increased portfolio ECL by ~4.6%

4. MODEL LIMITATIONS & ASSUMptions
   - Synthetic Data: Model trained on synthetically generated retail loan data.
   - Static LGD: LGD was mapped via LTV buckets; in production, a Beta regression model is recommended.
   - CCF Assumption: A fl